<a href="https://colab.research.google.com/github/h23yonsei/ECO4126_TinyGPT/blob/main/notebook_06_h23yonsei.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/h23yonsei/ECO4126_TinyGPT/refs/heads/main/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 64
dataset = NextTokenDataset(data, block_size)

# 데이터를 학습(80%)과 검증(20%)으로 분리합니다
# Overfitting 여부를 train/val loss 비교로 확인할 수 있습니다
n = int(0.8 * len(dataset))
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [n, len(dataset) - n])
loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

xb, yb = next(iter(loader))

## 1. Multi-Head Attention

In [ ]:
class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

## 2. Feedforward + Block

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.GELU(),  # GELU는 음수 입력을 부드럽게 처리해 gradient flow가 ReLU보다 좋습니다 (GPT-2 이후 표준)
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## 3. Tiny GPT

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size, bias=False)
        # Weight Tying: token_embedding과 lm_head의 가중치를 공유합니다
        # 입력 embedding과 출력 projection이 같은 벡터 공간을 사용해
        # 파라미터 수를 줄이고 성능을 높입니다 (GPT-2도 이 방식을 사용합니다)
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.blocks(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        return logits

model = TinyGPT(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 64, 68])


## 4. Training

In [ ]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        # Gradient Clipping: gradient 폭발을 방지합니다 (max_norm=1.0이 표준값)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

@torch.no_grad()
def eval_loss(model, loader, device):
    # Validation loss를 계산합니다 — gradient 계산 없이 inference만 수행합니다
    model.eval()
    total_loss, total_count = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyGPT(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
# CosineAnnealingLR: 학습 후반부로 갈수록 학습률을 서서히 줄여 수렴을 안정화합니다
# 고정 lr보다 마지막 수렴 품질이 높아집니다
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    val = eval_loss(model, val_loader, device)
    scheduler.step()
    print(f"epoch {epoch:2d} | train {train_loss:.4f} | val {val:.4f}")

epoch  0 | train 6.4455 | val 2.4399
epoch  1 | train 2.5289 | val 2.2772
epoch  2 | train 2.2898 | val 2.0969
epoch  3 | train 2.1388 | val 1.9734
epoch  4 | train 2.0227 | val 1.8710
epoch  5 | train 1.9336 | val 1.7805
epoch  6 | train 1.8520 | val 1.7063
epoch  7 | train 1.7837 | val 1.6393
epoch  8 | train 1.7218 | val 1.5856
epoch  9 | train 1.6733 | val 1.5323
epoch 10 | train 1.6273 | val 1.4858
epoch 11 | train 1.5861 | val 1.4490
epoch 12 | train 1.5502 | val 1.4125
epoch 13 | train 1.5188 | val 1.3841
epoch 14 | train 1.4899 | val 1.3606
epoch 15 | train 1.4664 | val 1.3281
epoch 16 | train 1.4396 | val 1.3096
epoch 17 | train 1.4218 | val 1.2908
epoch 18 | train 1.4060 | val 1.2712
epoch 19 | train 1.3844 | val 1.2556
epoch 20 | train 1.3707 | val 1.2352
epoch 21 | train 1.3531 | val 1.2263
epoch 22 | train 1.3426 | val 1.2090
epoch 23 | train 1.3274 | val 1.1965
epoch 24 | train 1.3162 | val 1.1823
epoch 25 | train 1.3025 | val 1.1686
epoch 26 | train 1.2921 | val 1.1596
e

## 5. Sampling

In [ ]:
@torch.no_grad()
def sample_gpt(model, block_size, stoi, itos, device,
               start_text="Dorothy", max_new_tokens=1000,
               temperature=1.0, top_k=None):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        # Temperature Scaling: 낮을수록 더 결정론적(반복적), 높을수록 더 다양한 생성
        logits = logits / temperature
        if top_k is not None:
            # Top-k Sampling: 확률 상위 k개의 token만 남기고 나머지는 -inf로 마스킹합니다
            # 낮은 확률의 이상한 token이 선택되는 것을 방지합니다
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[:, [-1]]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

# temperature=0.8, top_k=40: 적당히 결정론적이면서 다양성을 유지하는 표준 설정입니다
print(sample_gpt(model, block_size, stoi, itos, device,
                 start_text="Dorothy", max_new_tokens=1000,
                 temperature=0.8, top_k=40))

Dorothy sitting the Wicked Witch looked up to kill the Witch of the
East, and when you came out of Dorothy dress the end of the road long
her like a place was of tin, and very and made of my fierce was ready of tin.

Dorothy looked up the road rooms silver shoes. And asked Dorothy ask he Scarecrow and said:

“I don’t know,” said the Scarecrow. “But I shall never get to to the
others,” said the Scarecrow, “but you can I certainly fearing and
the Tin Woodman had at the straw days and said, “let us at it, and called
the Winkies were all lay down and pulled through a balloon. And lived in the
wolves who made the little in a heart, for her grews so fierce and that in her they
would be done it, and woman shall have down not let until you wear around. The road
for all the Winged Monkeys was grew sharp away a good-bye. What came to chase show
and flew air with a small short, and I could get back on with the cyclone road them.”

“What was shall go to the Emerald City?” asked the Scarecrow.

The